# Mamba: Linear-Time Sequence Modeling with Selective State Spaces

**Paper:** [arXiv:2312.00752](https://arxiv.org/abs/2312.00752) - Albert Gu, Tri Dao (2023)

This notebook implements a **simplified Selective State-Space Model (S6)** from scratch in PyTorch,
trains it on toy sequence tasks (selective copy and induction heads), and benchmarks its
speed and memory against a vanilla Transformer at increasing sequence lengths.

**Key idea:** Unlike prior SSMs (S4) which use fixed (time-invariant) parameters, Mamba makes
the SSM parameters B, C, and the step size delta **input-dependent**, allowing the model to
selectively propagate or forget information -- achieving content-based reasoning similar to
attention while retaining O(N) linear-time inference.

## 1. Imports and Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import random
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


## 2. Baseline: S4 State Space Model (LTI)

S4 is the predecessor to Mamba. It uses **fixed** parameters (A, B, C, dt) that do not
depend on the input -- making it Linear Time Invariant (LTI). This allows computation as
either a recurrence or a global convolution, but prevents content-based selection.

We implement it with a diagonal A matrix (S4D-Real initialization) for simplicity.

In [2]:
class S4Layer(nn.Module):
    '''Baseline LTI State Space Model (S4-style with diagonal A).
    Parameters are FIXED (not input-dependent).
    h_t = A_bar * h_{t-1} + B_bar * x_t ; y_t = C * h_t
    '''
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A).unsqueeze(0).expand(d_model, -1).contiguous())
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.D = nn.Parameter(torch.ones(d_model))
        self.dt = nn.Parameter(torch.randn(d_model) * 0.01 + 0.001)

    def forward(self, x):
        B, L, D = x.shape
        N = self.d_state
        A = -torch.exp(self.A_log)
        dt = torch.sigmoid(self.dt)
        dt_expanded = dt.unsqueeze(1)
        A_bar = torch.exp(dt_expanded * A)
        B_bar = dt_expanded * self.B
        h = torch.zeros(B, D, N, device=x.device)
        outputs = []
        for t in range(L):
            x_t = x[:, t, :].unsqueeze(2)
            h = A_bar.unsqueeze(0) * h + B_bar.unsqueeze(0) * x_t
            y_t = (h * self.C.unsqueeze(0)).sum(dim=2) + self.D
            outputs.append(y_t)
        y = torch.stack(outputs, dim=1)
        return y

## 3. Core: Selective State Space Model (S6) -- Mamba Key Innovation

The selective SSM makes **B, C, and delta input-dependent** (functions of the current token).
This breaks LTI but enables content-based selection. As the paper explains (Section 3.5.2),
a large delta resets the state and focuses on the current input, while a small delta keeps
the state and ignores the current input.

- **A** remains a fixed learnable parameter (HiPPO-initialized diagonal)
- **B_t, C_t, delta_t** are computed from x_t via linear projections
- **Discretization:** ZOH (simplified for diagonal A)
- **Computation:** Naive sequential recurrence (for clarity; real Mamba uses fused parallel scan)

In [3]:
class SelectiveSSM(nn.Module):
    '''S6: Selective State Space Model -- the core of Mamba.
    B, C, and delta are input-dependent (functions of x_t).
    A is fixed (learnable, HiPPO-initialized diagonal).
    '''
    def __init__(self, d_model, d_state=16, dt_rank=8):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A).unsqueeze(0).expand(d_model, -1).contiguous())
        self.x_proj = nn.Linear(d_model, dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_model, bias=True)
        nn.init.zeros_(self.dt_proj.weight)
        nn.init.constant_(self.dt_proj.bias, 0.001)
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        B, L, D = x.shape
        N = self.d_state
        R = self.dt_rank
        A = -torch.exp(self.A_log)
        x_proj = self.x_proj(x)
        delta_rank = x_proj[..., :R]
        B_seq = x_proj[..., R:R+N]
        C_seq = x_proj[..., R+N:]
        delta = F.softplus(self.dt_proj(delta_rank))
        h = torch.zeros(B, D, N, device=x.device)
        outputs = []
        for t in range(L):
            dt = delta[:, t, :]
            B_t = B_seq[:, t, :]
            C_t = C_seq[:, t, :]
            dt_exp = dt.unsqueeze(2)
            A_bar = torch.exp(dt_exp * A.unsqueeze(0))
            B_bar = dt_exp * B_t.unsqueeze(1)
            x_t = x[:, t, :].unsqueeze(2)
            h = A_bar * h + B_bar * x_t
            y_t = (h * C_t.unsqueeze(1)).sum(dim=2) + self.D
            outputs.append(y_t)
        y = torch.stack(outputs, dim=1)
        return y

## 4. Mamba Block

The Mamba block combines the selective SSM with a gated MLP in a single homogenous block:
- Input -> linear projection (expansion to 2D) -> split into gate and main branches
- Main branch: Conv1d -> SelectiveSSM
- Gate branch: SiLU activation
- Output = gate * ssm_output -> linear projection -> + residual

This replaces the interleaved (attention + MLP) blocks of Transformers with a single
repeated block type.

In [4]:
class MambaBlock(nn.Module):
    '''A single Mamba block: proj -> conv -> SiLU -> SSM -> gate -> proj + residual.'''
    def __init__(self, d_model, d_state=16, dt_rank=8, conv_kernel=4):
        super().__init__()
        self.d_model = d_model
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=conv_kernel,
                              padding=conv_kernel - 1, groups=d_model)
        self.ssm = SelectiveSSM(d_model, d_state, dt_rank)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        x_proj = self.in_proj(x)
        gate, main = x_proj.chunk(2, dim=-1)
        main = main.transpose(1, 2)
        main = self.conv(main)[:, :, :x.size(1)]
        main = main.transpose(1, 2)
        ssm_out = self.ssm(main)
        gate = F.silu(gate)
        out = gate * ssm_out
        out = self.out_proj(out)
        return out + residual

## 5. Full Mamba Language Model

In [5]:
class MambaLM(nn.Module):
    '''Mamba-based language model: embedding + Mamba blocks + output head.'''
    def __init__(self, vocab_size, d_model=64, n_layers=2, d_state=16, dt_rank=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state, dt_rank) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def forward(self, x):
        h = self.embedding(x)
        for layer in self.layers:
            h = layer(h)
        h = self.norm(h)
        logits = self.head(h)
        return logits

## 6. Baseline: Vanilla Transformer

A standard decoder-only Transformer with multi-head self-attention and MLP blocks.
This serves as our comparison point -- it has O(L^2) attention complexity.

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads=4, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x, mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask)
        x = x + attn_out
        h = self.norm2(x)
        x = x + self.ff(h)
        return x

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2, n_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embedding.weight
        self.d_model = d_model

    def forward(self, x):
        B, L = x.shape
        h = self.embedding(x) * math.sqrt(self.d_model)
        mask = torch.triu(torch.ones(L, L, device=x.device) * float('-inf'), diagonal=1)
        for layer in self.layers:
            h = layer(h, mask=mask)
        h = self.norm(h)
        logits = self.head(h)
        return logits

## 7. Toy Task 1: Selective Copy

The model must copy specific "marked" tokens from random positions to the end of the sequence.
This tests **content-based selection** -- the model must identify which tokens are special
and remember them while ignoring the rest.

In [7]:
def generate_selective_copy(batch_size, seq_len, vocab_size=16, n_markers=3, marker_token=0):
    '''Generate selective copy task data.
    Sequence: [random tokens with a few markers inserted at random positions]
    Target: copy the marker values to the last n_markers positions
    '''
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    y = x.clone()
    for b in range(batch_size):
        positions = sorted(random.sample(range(seq_len - n_markers * 2 - 1), n_markers))
        vals = []
        for p in positions:
            x[b, p] = marker_token
            val = torch.randint(1, vocab_size, (1,)).item()
            x[b, p + 1] = val
            vals.append(val)
        for i, v in enumerate(vals):
            y[b, seq_len - n_markers + i] = v
    return x, y

# Quick test
x, y = generate_selective_copy(4, 32, vocab_size=16, n_markers=3)
print(f"Input shape: {x.shape}")
print(f"Example input:  {x[0].tolist()}")
print(f"Example target: {y[0].tolist()}")

Input shape: torch.Size([4, 32])
Example input:  [0, 10, 2, 0, 12, 6, 11, 15, 11, 9, 14, 5, 6, 10, 7, 8, 6, 11, 13, 2, 0, 8, 14, 12, 15, 14, 12, 10, 8, 5, 13, 1]
Example target: [13, 3, 2, 5, 7, 6, 11, 15, 11, 9, 14, 5, 6, 10, 7, 8, 6, 11, 13, 2, 5, 12, 14, 12, 15, 14, 12, 10, 8, 10, 12, 8]


## 8. Toy Task 2: Induction Heads

The classic in-context retrieval task. A pattern A ... B A ? where the model
must predict B. This tests the model's ability to look back and find matching
context -- a capability that Transformers solve with induction heads in attention.

In [8]:
def generate_induction_heads(batch_size, seq_len, vocab_size=16):
    '''Generate induction head task data.
    Pattern: [random] + anchor + value + [random] + anchor + [?]
    The model must predict the token that followed the first anchor.
    '''
    half = seq_len // 2
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    anchors = torch.randint(1, vocab_size, (batch_size,))
    values = torch.randint(1, vocab_size, (batch_size,))
    pos1 = half - 5
    pos2 = half + 5
    x[:, pos1] = anchors
    x[:, pos1 + 1] = values
    x[:, pos2] = anchors
    y = x.clone()
    y[:, pos2 + 1] = values
    y[:, :pos2 + 1] = x[:, :pos2 + 1]
    return x, y, anchors, values, pos1, pos2

def induction_task(batch_size, seq_len, vocab_size):
    x, y, _, _, _, _ = generate_induction_heads(batch_size, seq_len, vocab_size)
    return x, y

# Quick test
x, y, anchors, values, p1, p2 = generate_induction_heads(4, 32)
print(f"Example: anchor={anchors[0].item()}, value={values[0].item()}")
print(f"pos1={p1}, pos2={p2}")
print(f"Input:  {x[0].tolist()}")
print(f"Target: {y[0].tolist()}")

Example: anchor=6, value=13
pos1=11, pos2=21
Input:  [15, 1, 6, 15, 11, 3, 6, 9, 14, 8, 8, 6, 13, 14, 13, 8, 7, 14, 1, 14, 12, 6, 14, 13, 14, 9, 15, 9, 13, 6, 11, 15]
Target: [15, 1, 6, 15, 11, 3, 6, 9, 14, 8, 8, 6, 13, 14, 13, 8, 7, 14, 1, 14, 12, 6, 13, 13, 14, 9, 15, 9, 13, 6, 11, 15]


## 9. Training Loop

In [9]:
def train_model(model, task_fn, epochs=50, batch_size=32, seq_len=64,
              vocab_size=16, lr=1e-3, task_name='task', verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        n_batches = 10
        for _ in range(n_batches):
            x, y = task_fn(batch_size, seq_len, vocab_size)
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / n_batches
        losses.append(avg_loss)
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f"  [{task_name}] Epoch {epoch:3d}/{epochs} -- Loss: {avg_loss:.4f}")
    return losses

D_MODEL = 64
D_STATE = 16
N_LAYERS = 2
VOCAB_SIZE = 16
EPOCHS = 50
SEQ_LEN = 64
DT_RANK = 8

print("Building models...")
mamba_model = MambaLM(VOCAB_SIZE, D_MODEL, N_LAYERS, D_STATE, DT_RANK).to(device)
transformer_model = TransformerLM(VOCAB_SIZE, D_MODEL, N_LAYERS, n_heads=4).to(device)
n_params_m = sum(p.numel() for p in mamba_model.parameters())
n_params_t = sum(p.numel() for p in transformer_model.parameters())
print(f"Mamba params:       {n_params_m:,}")
print(f"Transformer params: {n_params_t:,}")

Building models...


Mamba params:       35,072
Transformer params: 101,120


### Training on Selective Copy Task

In [10]:
print("=" * 60)
print("Training Mamba on Selective Copy")
print("=" * 60)
mamba_copy_losses = train_model(mamba_model, generate_selective_copy,
                                 epochs=EPOCHS, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                                 task_name='Mamba-Copy')

print()
print("=" * 60)
print("Training Transformer on Selective Copy")
print("=" * 60)
transformer_copy_losses = train_model(transformer_model, generate_selective_copy,
                                       epochs=EPOCHS, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                                       task_name='Transformer-Copy')

Training Mamba on Selective Copy


  [Mamba-Copy] Epoch   0/50 -- Loss: 6.9061


  [Mamba-Copy] Epoch  10/50 -- Loss: 0.5525


  [Mamba-Copy] Epoch  20/50 -- Loss: 0.5390


  [Mamba-Copy] Epoch  30/50 -- Loss: 0.5361


...

  [Transformer-Copy] Epoch  10/50 -- Loss: 0.6755


  [Transformer-Copy] Epoch  20/50 -- Loss: 0.6486


  [Transformer-Copy] Epoch  30/50 -- Loss: 0.6421


  [Transformer-Copy] Epoch  40/50 -- Loss: 0.6406


  [Transformer-Copy] Epoch  49/50 -- Loss: 0.6373


### Training on Induction Heads Task

In [11]:
mamba_model2 = MambaLM(VOCAB_SIZE, D_MODEL, N_LAYERS, D_STATE, DT_RANK).to(device)
transformer_model2 = TransformerLM(VOCAB_SIZE, D_MODEL, N_LAYERS, n_heads=4).to(device)

print("=" * 60)
print("Training Mamba on Induction Heads")
print("=" * 60)
mamba_ind_losses = train_model(mamba_model2, induction_task,
                                epochs=EPOCHS, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                                task_name='Mamba-Induction')

print()
print("=" * 60)
print("Training Transformer on Induction Heads")
print("=" * 60)
transformer_ind_losses = train_model(transformer_model2, induction_task,
                                      epochs=EPOCHS, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                                      task_name='Transformer-Induction')

Training Mamba on Induction Heads


  [Mamba-Induction] Epoch   0/50 -- Loss: 0.8592


  [Mamba-Induction] Epoch  10/50 -- Loss: 0.1216


  [Mamba-Induction] Epoch  20/50 -- Loss: 0.1177


  [Mamba-Induction] Epoch  30/50 -- Loss: 0.1170


...

  [Transformer-Induction] Epoch  10/50 -- Loss: 0.1261


  [Transformer-Induction] Epoch  20/50 -- Loss: 0.1154


  [Transformer-Induction] Epoch  30/50 -- Loss: 0.1140


  [Transformer-Induction] Epoch  40/50 -- Loss: 0.1151


  [Transformer-Induction] Epoch  49/50 -- Loss: 0.1141


## 10. Training Loss Visualization

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mamba_copy_losses, label='Mamba (S6)', linewidth=2)
axes[0].plot(transformer_copy_losses, label='Transformer', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Selective Copy Task')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(mamba_ind_losses, label='Mamba (S6)', linewidth=2)
axes[1].plot(transformer_ind_losses, label='Transformer', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cross-Entropy Loss')
axes[1].set_title('Induction Heads Task')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training loss plot saved.")

print(f"\nFinal losses (Selective Copy):  Mamba={mamba_copy_losses[-1]:.4f}, Transformer={transformer_copy_losses[-1]:.4f}")
print(f"Final losses (Induction Heads): Mamba={mamba_ind_losses[-1]:.4f}, Transformer={transformer_ind_losses[-1]:.4f}")

Training loss plot saved.

Final losses (Selective Copy):  Mamba=0.5381, Transformer=0.6373
Final losses (Induction Heads): Mamba=0.1131, Transformer=0.1141


## 11. Speed and Memory Benchmark

We compare Mamba and Transformer at increasing sequence lengths, measuring:
- **Forward + backward time** (averaged over multiple runs)
- **Peak GPU memory** (via torch.cuda.max_memory_allocated)

Expected: Mamba scales **linearly** O(L), Transformer scales **quadratically** O(L^2).

In [13]:
def benchmark_model(model_fn, seq_lengths, vocab_size=16, d_model=64,
                    batch_size=4, warmup=2, runs=3):
    results = {'seq_len': [], 'time_ms': [], 'memory_mb': []}
    for L in seq_lengths:
        model = model_fn().to(device)
        model.train()
        x = torch.randint(0, vocab_size, (batch_size, L), device=device)
        y = torch.randint(0, vocab_size, (batch_size, L), device=device)
        criterion = nn.CrossEntropyLoss()

        for _ in range(warmup):
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            if device.type == 'cuda':
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()

        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        times = []
        for _ in range(runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.time()
            times.append((t1 - t0) * 1000)

        avg_time = sum(times) / len(times)
        if device.type == 'cuda':
            peak_mem = torch.cuda.max_memory_allocated() / 1e6
        else:
            peak_mem = 0.0

        results['seq_len'].append(L)
        results['time_ms'].append(avg_time)
        results['memory_mb'].append(peak_mem)
        print(f"  L={L:5d} -- Time: {avg_time:8.1f} ms, Memory: {peak_mem:8.1f} MB")

        del model, x, y, logits, loss
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return results

SEQ_LENGTHS = [256, 512, 1024, 2048, 4096]
BENCH_VOCAB = 16
BENCH_D = 64
BENCH_BATCH = 4

print("=" * 60)
print("Benchmarking Mamba (S6)")
print("=" * 60)
mamba_bench = benchmark_model(
    lambda: MambaLM(BENCH_VOCAB, BENCH_D, N_LAYERS, D_STATE, DT_RANK),
    SEQ_LENGTHS, vocab_size=BENCH_VOCAB, d_model=BENCH_D, batch_size=BENCH_BATCH
)

print()
print("=" * 60)
print("Benchmarking Transformer")
print("=" * 60)
transformer_bench = benchmark_model(
    lambda: TransformerLM(BENCH_VOCAB, BENCH_D, N_LAYERS, n_heads=4),
    SEQ_LENGTHS, vocab_size=BENCH_VOCAB, d_model=BENCH_D, batch_size=BENCH_BATCH
)

Benchmarking Mamba (S6)


  L=  256 -- Time:    370.9 ms, Memory:     52.9 MB


  L=  512 -- Time:    692.2 ms, Memory:     85.2 MB


  L= 1024 -- Time:   1426.4 ms, Memory:    150.5 MB


  L= 2048 -- Time:   2893.3 ms, Memory:    280.1 MB


  L= 4096 -- Time:   5858.1 ms, Memory:    537.0 MB

Benchmarking Transformer
  L=  256 -- Time:      5.0 ms, Memory:     48.7 MB


  L=  512 -- Time:      5.3 ms, Memory:    119.1 MB
  L= 1024 -- Time:     15.8 ms, Memory:    383.3 MB


  L= 2048 -- Time:     54.5 ms, Memory:   1416.6 MB


  L= 4096 -- Time:    210.0 ms, Memory:   5496.4 MB


## 12. Benchmark Visualization

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].loglog(mamba_bench['seq_len'], mamba_bench['time_ms'], 'o-', label='Mamba (S6)', linewidth=2, markersize=8)
axes[0].loglog(transformer_bench['seq_len'], transformer_bench['time_ms'], 's-', label='Transformer', linewidth=2, markersize=8)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Time (ms) -- fwd+bwd')
axes[0].set_title('Speed: Mamba vs Transformer (log-log)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, which='both')

Ls = mamba_bench['seq_len']
ref_linear = [mamba_bench['time_ms'][0] * (l / Ls[0]) for l in Ls]
ref_quad = [transformer_bench['time_ms'][0] * (l / Ls[0])**2 for l in Ls]
axes[0].loglog(Ls, ref_linear, '--', color='blue', alpha=0.3, label='O(n) ref')
axes[0].loglog(Ls, ref_quad, '--', color='orange', alpha=0.3, label='O(n^2) ref')
axes[0].legend(fontsize=8)

if max(mamba_bench['memory_mb']) > 0 or max(transformer_bench['memory_mb']) > 0:
    axes[1].semilogy(mamba_bench['seq_len'], [max(m, 1) for m in mamba_bench['memory_mb']], 'o-', label='Mamba (S6)', linewidth=2, markersize=8)
    axes[1].semilogy(transformer_bench['seq_len'], [max(m, 1) for m in transformer_bench['memory_mb']], 's-', label='Transformer', linewidth=2, markersize=8)
    axes[1].set_xlabel('Sequence Length')
    axes[1].set_ylabel('Peak Memory (MB)')
    axes[1].set_title('Memory: Mamba vs Transformer')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Memory benchmark\nrequires GPU', ha='center', va='center', transform=axes[1].transAxes, fontsize=14)
    axes[1].set_title('Memory: Mamba vs Transformer')

plt.tight_layout()
plt.savefig('benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("Benchmark plot saved.")

Benchmark plot saved.


## 13. Results Summary

In [15]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

print(f"\n{'Task':<25} {'Mamba Loss':>12} {'Transformer Loss':>18}")
print("-" * 55)
print(f"{'Selective Copy':<25} {mamba_copy_losses[-1]:>12.4f} {transformer_copy_losses[-1]:>18.4f}")
print(f"{'Induction Heads':<25} {mamba_ind_losses[-1]:>12.4f} {transformer_ind_losses[-1]:>18.4f}")

print(f"\n{'Seq Len':>8} {'Mamba (ms)':>12} {'Transformer (ms)':>18} {'Speedup':>10}")
print("-" * 55)
for i, L in enumerate(SEQ_LENGTHS):
    speedup = transformer_bench['time_ms'][i] / max(mamba_bench['time_ms'][i], 0.001)
    print(f"{L:>8} {mamba_bench['time_ms'][i]:>12.1f} {transformer_bench['time_ms'][i]:>18.1f} {speedup:>9.2f}x")

print(f"\n{'Seq Len':>8} {'Mamba (MB)':>12} {'Transformer (MB)':>18}")
print("-" * 42)
for i, L in enumerate(SEQ_LENGTHS):
    print(f"{L:>8} {mamba_bench['memory_mb'][i]:>12.1f} {transformer_bench['memory_mb'][i]:>18.1f}")

print()
print("KEY OBSERVATIONS:")
print("1. LOSS ONLY: Final losses are close (copy 0.5381 vs 0.6373, induction")
print("   0.1131 vs 0.1141). No accuracy is measured, so this run does not show")
print("   that either task is solved, and the S4 baseline was not trained here.")
print("2. SPEED: This naive-loop Mamba is 28x to 131x SLOWER than the Transformer")
print("   at every length tested. Its time grows about linearly (2x per doubling);")
print("   the Transformer's grows faster (3.9x from 2048 to 4096) but stays lower.")
print("3. MEMORY EFFICIENCY: From L=512 on, Mamba uses less memory (537 MB vs")
print("   5496 MB at L=4096, about 10x less): there is no L x L attention matrix.")
print("4. SIMPLIFICATION: This notebook uses a naive Python-loop recurrence, not")
print("   the hardware-aware fused parallel scan. Real Mamba uses CUDA-fused kernels")
print("   for much greater speed; only the growth rates are comparable here.")
print("5. ARCHITECTURE: Mamba replaces both attention AND MLP blocks with a single")
print("   homogenous block type, simplifying the architecture.")

RESULTS SUMMARY

Task                        Mamba Loss   Transformer Loss
-------------------------------------------------------
Selective Copy                  0.5381             0.6373
Induction Heads                 0.1131             0.1141

 Seq Len   Mamba (ms)   Transformer (ms)    Speedup
-------------------------------------------------------
     256        370.9                5.0      0.01x
     512        692.2                5.3      0.01x
    1024       1426.4               15.8      0.01x
    2048       2893.3               54.5      0.02x
    4096       5858.1              210.0      0.04x

 Seq Len   Mamba (MB)   Transformer (MB)
------------------------------------------
     256         52.9               48.7
     512         85.2              119.1
    1024        150.5              383.3
    2048        280.1             1416.6
    4096        537.0             5496.4

KEY OBSERVATIONS:
1. LOSS ONLY: Final losses are close (copy 0.5381 vs 0.6373, induction
   0

## 14. Conclusion

This notebook demonstrated the core mechanism of Mamba -- **selective state-space models**:

1. **Selection:** Making SSM parameters input-dependent (B, C, delta are functions of x_t) enables
   content-based reasoning, which fixed LTI models like S4 cannot do.

2. **Efficiency:** Despite losing the convolutional computation of LTI models, the selective
   recurrence still scales linearly in sequence length, while Transformer attention scales
   quadratically.

3. **Simplicity:** The Mamba block is a single homogenous block that replaces both attention
   and MLP blocks, and here it reached training losses similar to the Transformer on the toy tasks.

The real Mamba paper achieves state-of-the-art results on language, audio, and genomics
benchmarks, with Mamba-3B outperforming Transformers of the same size and matching Transformers twice its size. The hardware-aware
parallel scan (not implemented here for clarity) is key to making this practical at scale.

**Paper:** [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752)

### What the run above shows

- **Toy tasks (loss only, no accuracy):** selective copy, Mamba 0.5381 vs Transformer 0.6373; induction heads, Mamba 0.1131 vs Transformer 0.1141. The two models end at similar losses. Without an accuracy number we cannot say either task is solved, and the `S4Layer` baseline was never trained, so there is no S4 vs S6 comparison.
- **Speed:** this naive Python-loop Mamba was **28× to 131× slower** than the Transformer at every length (370.9 vs 5.0 ms at L=256; 5858.1 vs 210.0 ms at L=4096). Mamba's time roughly doubled per doubling of length (linear); the Transformer's grew 3.9× from 2048 to 4096. The paper's speed comes from the fused CUDA scan, which is not used here.
- **Memory:** from L=512 on, Mamba used less peak memory: 537.0 MB vs 5496.4 MB at L=4096 (about 10× less). This matches the paper's claim that there is no L × L attention matrix to store.
- The loss and benchmark plots were saved to files but not displayed in this run.